In [15]:
import json
import folium
from urllib.request import urlopen
from urllib.parse import urlencode

# 別の都市名に変えてみよう
city = input("都市名を入力してください (例: Tokyo, New York, Paris): ")
# 取得する候補の数
match_count = 1

# 地名の表示言語
language = "en"

# 通信を待つ最大秒数
timeout = 10

# ② 都市名を検索用のURLに入れ、場所の情報を取得する
url = "https://geocoding-api.open-meteo.com/v1/search?" + urlencode({
    "name": city, "count": match_count, "language": language
})
with urlopen(url, timeout=timeout) as response:
    locations = json.load(response)

place = locations["results"][0]
place["name"], place.get("admin1"), place["country"]

KeyError: 'results'

In [16]:
lat = place["latitude"]

lon = place["longitude"]

past_days = 0
# 今日を含めて何日分を取得するか
forecast_days = 7

timezone = "auto"

daily_fields = (
    "precipitation_probability,"
)

timeout = 10

url = "https://api.open-meteo.com/v1/forecast?" + urlencode({
    "latitude": lat, "longitude": lon,
    "daily": daily_fields,
    "past_days": past_days, "forecast_days": forecast_days,
    "timezone": timezone
})

# ③ 返ってきたJSONを辞書として読み込む
with urlopen(url, timeout=timeout) as response:
    weather = json.load(response)

weather_map = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10
)

# 中心都市を表示
folium.Marker(
    [center_lat, center_lon],
    popup=city_name,
    tooltip="調査対象都市"
).add_to(weather_map)

# =====================================
# 5. 降水確率を色分け
# =====================================

def get_color(probability):

    if probability <= 20:
        return "blue"
    elif probability <= 40:
        return "lightblue"
    elif probability <= 60:
        return "yellow"
    elif probability <= 80:
        return "orange"
    else:
        return "red"


# =====================================
# 6. 時間ごとの地図レイヤーを作成
# =====================================

# 複数地点のデータを取得
if isinstance(weather_data, dict):
    weather_data = [weather_data]

# 最初の地点の時刻を使用
times = weather_data[0]["hourly"]["time"]

for t, time in enumerate(times):

    # 時間ごとのレイヤー
    layer = folium.FeatureGroup(
        name=time,
        show=(t == 0)
    )

    for i, point in enumerate(weather_data):

        probability = point["hourly"][
            "precipitation_probability"
        ][t]

        if probability is None:
            continue

        lat = latitudes[i]
        lon = longitudes[i]

        folium.CircleMarker(
            location=[lat, lon],
            radius=13,
            color=get_color(probability),
            fill=True,
            fill_color=get_color(probability),
            fill_opacity=0.65,
            popup=f"""
            時刻: {time}<br>
            降水確率: {probability}%
            """
        ).add_to(layer)

    layer.add_to(weather_map)

# =====================================
# 7. 時間切り替え機能
# =====================================

folium.LayerControl(
    collapsed=False
).add_to(weather_map)

# =====================================
# 8. 凡例を追加
# =====================================

legend_html = """
<div style="
position: fixed;
bottom: 30px;
left: 30px;
width: 180px;
background-color: white;
border: 2px solid grey;
z-index: 9999;
padding: 10px;
font-size: 14px;
">
<b>降水確率</b><br>
<span style="color:blue">●</span> 0～20%<br>
<span style="color:lightblue">●</span> 21～40%<br>
<span style="color:gold">●</span> 41～60%<br>
<span style="color:orange">●</span> 61～80%<br>
<span style="color:red">●</span> 81～100%
</div>
"""

weather_map.get_root().html.add_child(
    folium.Element(legend_html)
)

# =====================================
# 9. HTMLファイルとして保存
# =====================================

filename = "weather_map.html"

weather_map.save(filename)

print("地図を作成しました！")
print(f"{filename} をブラウザで開いてください。")

HTTPError: HTTP Error 400: Bad Request

In [4]:

import requests
import folium
import math

# =====================================
# 1. 調べたい都市を入力
# =====================================

city = input("調べたい都市名を入力してください（例：Tokyo）: ")

# Open-Meteo Geocoding APIで都市の緯度・経度を取得
geo_url = "https://geocoding-api.open-meteo.com/v1/search"

geo_params = {
    "name": city,
    "count": 1,
    "language": "ja",
    "format": "json"
}

geo_response = requests.get(geo_url, params=geo_params)
geo_data = geo_response.json()

if not geo_data.get("results"):
    print("都市が見つかりませんでした。")
    raise SystemExit

location = geo_data["results"][0]

center_lat = location["latitude"]
center_lon = location["longitude"]
city_name = location["name"]

print(f"{city_name}の緯度: {center_lat}")
print(f"{city_name}の経度: {center_lon}")

# =====================================
# 2. 周辺25地点の座標を作成
# =====================================

latitudes = []
longitudes = []

# 約5km間隔で5×5の地点を作成
spacing_km = 5

for i in range(-2, 3):
    for j in range(-2, 3):

        lat = center_lat + (i * spacing_km / 111)

        lon = center_lon + (
            j * spacing_km /
            (111 * math.cos(math.radians(center_lat)))
        )

        latitudes.append(lat)
        longitudes.append(lon)

# =====================================
# 3. Open-Meteoから降水確率を取得
# =====================================

weather_url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": ",".join(map(str, latitudes)),
    "longitude": ",".join(map(str, longitudes)),
    "hourly": "precipitation_probability",
    "forecast_days": 2,
    "timezone": "auto"
}

response = requests.get(weather_url, params=params)
weather_data = response.json()

# =====================================
# 4. 地図を作成
# =====================================

weather_map = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri, HERE, Garmin, USGS, Esri Japan, METI"
)

# 中心都市を表示
folium.Marker(
    [center_lat, center_lon],
    popup=city_name,
    tooltip="調査対象都市"
).add_to(weather_map)

# =====================================
# 5. 降水確率を色分け
# =====================================

def get_color(probability):

    if probability <= 20:
        return "blue"
    elif probability <= 40:
        return "lightblue"
    elif probability <= 60:
        return "yellow"
    elif probability <= 80:
        return "orange"
    else:
        return "red"


# =====================================
# 6. 時間ごとの地図レイヤーを作成
# =====================================

# 複数地点のデータを取得
if isinstance(weather_data, dict):
    weather_data = [weather_data]

# 最初の地点の時刻を使用
times = weather_data[0]["hourly"]["time"]

for t, time in enumerate(times):

    # 時間ごとのレイヤー
    layer = folium.FeatureGroup(
        name=time,
        show=(t == 0)
    )

    for i, point in enumerate(weather_data):

        probability = point["hourly"][
            "precipitation_probability"
        ][t]

        if probability is None:
            continue

        lat = latitudes[i]
        lon = longitudes[i]

        folium.CircleMarker(
            location=[lat, lon],
            radius=13,
            color=get_color(probability),
            fill=True,
            fill_color=get_color(probability),
            fill_opacity=0.65,
            popup=f"""
            時刻: {time}<br>
            降水確率: {probability}%
            """
        ).add_to(layer)

    layer.add_to(weather_map)

# =====================================
# 7. 時間切り替え機能
# =====================================

folium.LayerControl(
    collapsed=False
).add_to(weather_map)

# =====================================
# 8. 凡例を追加
# =====================================

legend_html = """
<div style="
position: fixed;
bottom: 30px;
left: 30px;
width: 180px;
background-color: white;
border: 2px solid grey;
z-index: 9999;
padding: 10px;
font-size: 14px;
">
<b>降水確率</b><br>
<span style="color:blue">●</span> 0～20%<br>
<span style="color:lightblue">●</span> 21～40%<br>
<span style="color:gold">●</span> 41～60%<br>
<span style="color:orange">●</span> 61～80%<br>
<span style="color:red">●</span> 81～100%
</div>
"""

weather_map.get_root().html.add_child(
    folium.Element(legend_html)
)

# =====================================
# 9. HTMLファイルとして保存
# =====================================

filename = "weather_map.html"

weather_map.save(filename)

print("地図を作成しました！")
print(f"{filename} をブラウザで開いてください。")


柏市の緯度: 35.86224
柏市の経度: 139.97733
地図を作成しました！
weather_map.html をブラウザで開いてください。
